## **Progetto – Rete di trasporti e inferenza a mondo chiuso (CWA)**

In questo progetto dovrai costruire una base di conoscenza che rappresenta una rete di voli diretti tra città, utilizzando OWL e la libreria owlready2. Le informazioni presenti nella KB saranno incomplete: conoscerai solo i voli esistenti, e non quelli mancanti. Per dedurre l’assenza di un volo, applicherai il principio del Closed World Assumption (CWA): tutto ciò che non è noto, si assume falso.

- Definire una base di conoscenza OWL con le classi *Città* e *Volo*;
- Modellare le proprietà:
  - *volo_diretto(a, b)*: esiste un volo diretto da a a b;
  - *volo_indiretto(a, b)*: esiste un percorso da a a b con almeno uno scalo;
- Costruire una procedura automatica di **forward chaining** che:
  - derivi automaticamente i voli indiretti tramite regole del tipo:
    - *se esiste un volo diretto da A a B e da B a C, allora esiste un volo indiretto da A a C;*
  - generi tutte le coppie di città possibili, e per ognuna verifichi:
    - se il volo diretto è presente;
    - se non è presente, lo segni come negato (¬volo) assumendo il **CWA**;
-  Implementare **query automatiche** per rispondere a domande del tipo:
    - *“Esiste un volo diretto da X a Y?”*
    - *“Esiste un volo indiretto tra X e Y?”*
    - *“È noto che non esista alcun volo tra X e Y?”*
- Dimostrare che l’aggiunta di un singolo volo diretto può modificare le inferenze indirette (es. sbloccare un collegamento indiretto tra città) evidenziando come il ragionamento sia **non monotono**

Il progetto dovrà contenere almeno **4 città**, **4 voli diretti** e presentare esempi in cui sia possibile:
- inferire voli indiretti;
- dedurre l’assenza di voli sulla base del CWA;
- verificare le conseguenze di un aggiornamento alla KB.

### Obiettivi del progetto

In [ ]:
from owlready2 import *
import itertools
from IPython.display import Image, display

In [ ]:
onto = get_ontology("http://example.org/transport_network.owl")
onto

In [ ]:
with onto:
    #Classi
    class Citta(Thing): pass
    class Volo(Thing): pass
    
    #Proprietà oggetto
    class volo_diretto(ObjectProperty):
        domain = [Citta]
        range = [Citta]
    
    class volo_indiretto(ObjectProperty):
        domain = [Citta]
        range = [Citta]
    
    class nessun_volo(ObjectProperty):
        domain = [Citta]
        range = [Citta]

In [ ]:
with onto:
    #Città
    atene = Citta("Atene")
    barcellona = Citta("Barcellona")
    londra = Citta("Londra")
    luqa = Citta("Luqa")
    palermo = Citta("Palermo")
    parigi = Citta("Parigi")

    #Voli diretti
    atene.volo_diretto = [parigi]
    barcellona.volo_diretto = [atene, parigi]
    londra.volo_diretto = [palermo]
    luqa.volo_diretto = [londra]
    palermo.volo_diretto = [luqa, barcellona]
    parigi.volo_diretto = [londra]

In [ ]:
#Crea la lista con le città
lista_citta = list(onto.Citta.instances())

In [ ]:
#Mostra le città
def stampa_citta():
    print("=== Istanze di città create ===")
    for citta in lista_citta: print(f"- {citta.name}")

In [ ]:
#Mostra i voli iniziali
def stampa_voli():
    print("\n=== Voli diretti iniziali ===")
    for citta in lista_citta:
        voli_diretti = getattr(citta, 'volo_diretto', [])
        #Controlla se la città ha effettivamente voli diretti
        if voli_diretti:
            destinazioni = [d.name for d in voli_diretti]
            print(f"- {citta.name} -> {', '.join(destinazioni)}")

In [ ]:
#Metodo di supporto per controllare se l'origine ha un volo diretto verso la destinazione
def ha_volo_diretto(origine, destinazione): return destinazione in getattr(origine, 'volo_diretto', [])

In [ ]:
def forward_chaining():
    print("\n=== FORWARD CHAINING - Derivazione dei voli indiretti ===")
    for origine in lista_citta:
        for destinazione in lista_citta:

            #Verifica se esiste già un volo indiretto
            voli_indiretti_attuali = getattr(origine, 'volo_indiretto', [])
            if destinazione in voli_indiretti_attuali: continue
                
            #Cerca percorsi indiretti tramite città intermedie
            voli_diretti_da_a = getattr(origine, 'volo_diretto', [])
            for citta_intermedia in voli_diretti_da_a:
                #Verifica l'esistenza di un volo diretto da citta_intermedia a destinazione
                if ha_volo_diretto(citta_intermedia, destinazione):
                    #Sappiamo se esiste o meno un volo indiretto origine -> destinazione via citta_intermedia
                    if not hasattr(origine, 'volo_indiretto'): origine.volo_indiretto = []
                    origine.volo_indiretto.append(destinazione)
                    print(f"Inferenza: volo indiretto nella tratta {origine.name} -> {destinazione.name} via {citta_intermedia.name}")
                    break

In [ ]:
def trova_citta(nome_citta):
    #Restituisce la citta partendo dal nome, se presente nella KB
    for citta in lista_citta:
        if citta.name == nome_citta: return citta
    return None

In [ ]:
def query_volo_diretto(nome_origine, nome_destinazione):
    #Esiste un volo diretto da X a Y?
    origine = trova_citta(nome_origine)
    destinazione = trova_citta(nome_destinazione)
    
    #Verifica se la citta o la destinazione sono nella KB
    if not origine or not destinazione: return f"Città non trovate: {nome_origine} o {nome_destinazione}"
    
    #Verifica la query
    if ha_volo_diretto(origine, destinazione): return f"- Sì, esiste un volo diretto nella tratta {nome_origine} -> {nome_destinazione}"
    else: return f"- No, non esiste un volo diretto nella tratta {nome_origine} -> {nome_destinazione}"

In [ ]:
def query_volo_indiretto(nome_origine, nome_destinazione):
    #Esiste un volo indiretto tra X e Y?
    origine = trova_citta(nome_origine)
    destinazione = trova_citta(nome_destinazione)
    
    #Verifica se la citta o la destinazione sono nella KB
    if not origine or not destinazione: return f"Città non trovate: {nome_origine} o {nome_destinazione}"
    
    #Verifica la query
    voli_indiretti = getattr(origine, 'volo_indiretto', [])
    if destinazione in voli_indiretti: return f"- Sì, esiste un volo indiretto nella tratta {nome_origine} -> {nome_destinazione}"
    else: return f"- No, non esiste un volo indiretto nella tratta {nome_origine} -> {nome_destinazione}"

In [ ]:
def query_nessun_volo(nome_origine, nome_destinazione):
    #È noto che non esista alcun volo tra X e Y?
    origine = trova_citta(nome_origine)
    destinazione = trova_citta(nome_destinazione)
    
    #Verifica se la citta o la destinazione sono nella KB
    if not origine or not destinazione: return f"Città non trovate: {nome_origine} o {nome_destinazione}"
    
    #Verifica la query
    nessun_volo = getattr(origine, 'nessun_volo', [])
    if destinazione in nessun_volo: return f"- È noto (tramite CWA) che non esiste alcun volo nella tratta {nome_origine} -> {nome_destinazione}"
    else: return f"- Non è noto che non esista un volo nella tratta {nome_origine} -> {nome_destinazione}"

In [ ]:
def applica_CWA():
    print("\n=== APPLICAZIONE DELLA CLOSED WORLD ASSUMPTION ===")
    
    #Crea le coppie di tutte le città, con A,B =/= B,A
    coppie_citta = list(itertools.permutations(lista_citta, 2))
    
    for origine, destinazione in coppie_citta:
        #Controlla se vi è un volo diretto tra origine e destinazione
        ha_diretto = ha_volo_diretto(origine, destinazione)
        
        #Se non è presente un volo diretto, controlla se l'origine presenta voli indiretti
        voli_indiretti = getattr(origine, 'volo_indiretto', [])
        ha_indiretto = destinazione in voli_indiretti
        
        #Se l'origine non ha ne voli diretti ne indiretti, allora non ha voli verso destinazione
        if not ha_diretto and not ha_indiretto:
            if not hasattr(origine, 'nessun_volo'): origine.nessun_volo = []
            origine.nessun_volo.append(destinazione)
            print(f"CWA eseguita: nessun volo disponibile nella tratta {origine.name} -> {destinazione.name}")

In [ ]:
def stampa_stato_base_conoscenza():
    #Mostra lo stato attuale della KB
    for citta in lista_citta:
        print(f"\nCittà: {citta.name}")
        
        #Voli diretti
        voli_diretti = getattr(citta, 'volo_diretto', [])
        #Controlla se la città ha effettivamente voli diretti
        if voli_diretti:
            nomi_citta_voli_diretti = [citta.name for citta in voli_diretti]
            print(f"Voli diretti verso: {', '.join(nomi_citta_voli_diretti)}")
        
        #Voli indiretti
        voli_indiretti = getattr(citta, 'volo_indiretto', [])
        #Controlla se la città ha effettivamente voli indiretti
        if voli_indiretti:
            nomi_citta_voli_indiretti = [citta.name for citta in voli_indiretti]
            print(f"Voli indiretti verso: {', '.join(nomi_citta_voli_indiretti)}")
        
        #Voli non disponibili (CWA)
        nessun_volo = getattr(citta, 'nessun_volo', [])
        #Controlla se la città non ha effettivamente voli disponibili
        if nessun_volo:
            nomi_citta_nessun_volo = [citta.name for citta in nessun_volo]
            print(f"Nessun volo verso (CWA) verso: {', '.join(nomi_citta_nessun_volo)}")


In [ ]:
def reset_voli_indiretti_e_nulli():
    #Resetta le informazioni riguardanti i voli indiretti e nulli
    for citta in lista_citta:
        if hasattr(citta, 'volo_indiretto'): citta.volo_indiretto = []
        if hasattr(citta, 'nessun_volo'): citta.nessun_volo = []

In [ ]:
def ragionamento_non_monotono():
    #Dimostrazione di un ragionamento non monotono dopo l'aggiunta di un viaggio diretto da Atene a Palermo
    print("\n=== DIMOSTRAZIONE DEL RAGIONAMENTO NON MONOTONO ===")

    #Resetta le inferenze fatte con la FC e con la CWA, poi le riesegue
    print("\n0. STATO DI origine:\nReset delle inferenze fatte con la Forward Chaining e con la Closed World Assumption per avere una KB pulita")
    reset_voli_indiretti_e_nulli()
    print("Riesecuzione della Forward Chaining e della Closed World Assumption")
    forward_chaining()
    applica_CWA()

    #Controllo dell'esistenza di un volo da Atene a Palermo
    print("\n1. STATO INIZIALE:\nEsiste un volo (diretto o indiretto) da Atene a Palermo?")
    print(query_volo_diretto("Atene", "Palermo"))
    print(query_volo_indiretto("Atene", "Palermo"))
    print(query_nessun_volo("Atene", "Palermo"))
    
    #Aggiunta di un nuovo volo diretto da Atene a Palermo
    print("\n2. AGGIUNTA DI NUOVO VOLO DIRETTO: Atene -> Palermo")
    atene = trova_citta("Atene")
    palermo = trova_citta("Palermo")
    if palermo not in atene.volo_diretto: atene.volo_diretto.append(palermo)
    print("Volo Atene -> Palermo aggiunto alla KB")

    #Reset delle inferenze dopo l'aggiunta del volo
    reset_voli_indiretti_e_nulli()
    
    #Aggiornamento della KB dopo l'aggiunta del volo diretto
    print("\n3. STATO DELLA KNOWLEDGE BASE DOPO L'AGGIORNAMENTO:")
    forward_chaining()
    applica_CWA()

    print(query_volo_diretto("Atene", "Palermo"))
    print(query_volo_indiretto("Atene", "Palermo"))

    print("\n4. CONCLUSIONE:\nL'aggiunta di un singolo volo diretto ha modificato lo stato della KB, dimostrando che il ragionamento è non monotono.")

In [ ]:
def spaziatura(): print("-" * 80)

In [ ]:
def main():
    spaziatura()
    stampa_citta()
    stampa_voli()
    spaziatura()
    display(Image(filename='tratta_progetto3_rkk.png', width = 600))
    spaziatura()
    print("\n=== STATO INIZIALE DELLA RETE DI TRASPORTI ===")
    stampa_stato_base_conoscenza()
    spaziatura()
    forward_chaining()
    spaziatura()
    applica_CWA()
    spaziatura()
    print("\n=== STATO FINALE DELLA RETE DI TRASPORTI (DOPO LE INFERENZE) ===")
    stampa_stato_base_conoscenza()
    spaziatura()
    print("\n=== ESEMPI DI QUERY AUTOMATICHE ===")
    esempi_query = [("Palermo", "Londra"), ("Atene", "Luqa"), ("Barcellona", "Londra"), ("Parigi", "Atene")]
    for origine, destinazione in esempi_query:
        print(f"\n- Query per {origine} -> {destinazione}")
        print(query_volo_diretto(origine, destinazione))
        print(query_volo_indiretto(origine, destinazione))
        print(query_nessun_volo(origine, destinazione))
    spaziatura()
    ragionamento_non_monotono()
    spaziatura()
    print("\n=== STATO FINALE DELLA RETE DI TRASPORTI (DOPO DIMOSTRAZIONE DI NON MONOTONIA) ===")
    stampa_stato_base_conoscenza()

In [ ]:
if __name__ == "__main__": main()